# Member 2 — Fusion Model v3: ResNet-18 + BioBERT

**Why this combination:**
- **Image encoder**: ResNet-18 (layer4 unfrozen) — ImageNet pretraining generalises across all medical image modalities (X-ray, MRI, CT, ultrasound). CheXNet was dropped because it is chest-X-ray-only and this dataset contains mixed modalities.
- **Text encoder**: BioBERT (`dmis-lab/biobert-base-cased-v1.2`) — BERT pretrained on PubMed + PMC; understands medical terminology better than CLIP's general-purpose text encoder.
- **Text pooling**: mean pooling over last hidden states (better than `pooler_output` for classification).
- **Fusion head**: LayerNorm on each stream before concat prevents scale mismatch between the two feature spaces.

**Architecture:**
```
Image (224x224 RGB) -> ResNet-18 (layer4 unfrozen) -> 512-dim -> LayerNorm
Question text       -> BioBERT  (mean pool)         -> 768-dim -> LayerNorm
Concat 1280-dim -> Linear(256) -> ReLU -> Dropout(0.4) -> Linear(1)
```

In [1]:
# Cell 1: Install dependencies
!pip install -q transformers datasets scikit-learn Pillow tqdm

In [2]:
# Cell 2: Imports
import io, os, copy
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.models as models
import torchvision.transforms as transforms

from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [3]:
# Cell 3: Load and filter dataset to binary (yes/no) samples
dataset = load_dataset("robailleo/medical-vision-llm-dataset")

def normalize_text(x):
    return str(x).strip().lower() if x is not None else ""

def is_binary(answer):
    return normalize_text(answer) in {"yes", "no"}

def label_from_answer(answer):
    return 1 if normalize_text(answer) == "yes" else 0

def filter_split(split_data):
    rows = []
    for row in split_data:
        answer = row.get("answer") or row.get("Answer") or ""
        if is_binary(answer):
            question = row.get("question") or row.get("Question") or ""
            rows.append({
                "question": question,
                "answer":   answer,
                "label":    label_from_answer(answer),
                "image":    row.get("image") or row.get("Image"),
            })
    return pd.DataFrame(rows)

train_df = filter_split(dataset["train"])
val_df   = filter_split(dataset["validation"])
train_df["split"] = "train"
val_df["split"]   = "validation"

print(f"Train size: {len(train_df)},  Val size: {len(val_df)}")
print("Label distribution (train):\n", train_df["label"].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/146M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/149M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/153M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/159M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/155M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3834 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/959 [00:00<?, ? examples/s]

Train size: 750,  Val size: 190
Label distribution (train):
 label
0    379
1    371
Name: count, dtype: int64


In [4]:
# Cell 4: Image preprocessing — standard ImageNet / ResNet pipeline
IMAGE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def open_pil_image(image_value):
    """Open PIL Image from HuggingFace dataset value."""
    if isinstance(image_value, Image.Image):
        return image_value
    if isinstance(image_value, dict):
        if image_value.get("bytes") is not None:
            return Image.open(io.BytesIO(image_value["bytes"]))
        if image_value.get("path") is not None:
            return Image.open(image_value["path"])
    return None

# Sanity check
sample = IMAGE_TRANSFORM(open_pil_image(train_df.iloc[0]["image"]))
print(f"Image tensor — shape: {sample.shape},  min: {sample.min():.3f},  max: {sample.max():.3f}")

Image tensor — shape: torch.Size([3, 224, 224]),  min: -2.118,  max: 2.623


In [5]:
# Cell 5: BioBERT tokenizer
BIOBERT_MODEL = "dmis-lab/biobert-base-cased-v1.2"
MAX_TEXT_LEN  = 128

biobert_tokenizer = AutoTokenizer.from_pretrained(BIOBERT_MODEL)
print("BioBERT tokenizer loaded. Vocab size:", biobert_tokenizer.vocab_size)

# Quick test
sample_q = train_df.iloc[0]["question"]
enc = biobert_tokenizer(sample_q, padding="max_length", truncation=True, max_length=MAX_TEXT_LEN)
print(f"Sample question : '{sample_q}'")
print(f"input_ids length: {len(enc['input_ids'])}")

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

BioBERT tokenizer loaded. Vocab size: 28996
Sample question : 'does this represent infectious process?'
input_ids length: 128


In [6]:
# Cell 6: Dataset class
class FusionDataset(Dataset):
    """
    Each __getitem__ returns:
        image      : FloatTensor (3, 224, 224)  — ImageNet normalised
        input_ids  : LongTensor  (MAX_TEXT_LEN,)
        attn_mask  : LongTensor  (MAX_TEXT_LEN,)
        label      : LongTensor  scalar (0 or 1)
    """
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.df         = dataframe.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Image
        img = open_pil_image(row["image"])
        if img is None:
            raise ValueError(f"Could not open image at index {idx}")
        image = IMAGE_TRANSFORM(img)

        # Text
        enc = self.tokenizer(
            str(row["question"]),
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
        )
        input_ids = torch.tensor(enc["input_ids"],      dtype=torch.long)
        attn_mask = torch.tensor(enc["attention_mask"], dtype=torch.long)

        # Label
        label = torch.tensor(int(row["label"]), dtype=torch.long)

        return {"image": image, "input_ids": input_ids,
                "attn_mask": attn_mask, "label": label}


BATCH_SIZE = 16

train_dataset = FusionDataset(train_df, biobert_tokenizer, MAX_TEXT_LEN)
val_dataset   = FusionDataset(val_df,   biobert_tokenizer, MAX_TEXT_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)},  Val batches: {len(val_loader)}")

# Verify batch shapes
batch = next(iter(train_loader))
print("image     :", batch["image"].shape)
print("input_ids :", batch["input_ids"].shape)
print("attn_mask :", batch["attn_mask"].shape)
print("label     :", batch["label"].shape)

Train batches: 47,  Val batches: 12
image     : torch.Size([16, 3, 224, 224])
input_ids : torch.Size([16, 128])
attn_mask : torch.Size([16, 128])
label     : torch.Size([16])


In [7]:
# Cell 7: Model — ResNet-18 + BioBERT fusion
#
# ResNet-18 (layer4 unfrozen) -> 512-dim -> LayerNorm
# BioBERT   (all frozen)      -> 768-dim -> LayerNorm  (mean pooled)
# Concat 1280 -> Linear(256) -> ReLU -> Dropout(0.4) -> Linear(1)

class ResNetBioBERTFusion(nn.Module):
    def __init__(self, biobert_model_name=BIOBERT_MODEL, dropout=0.4):
        super().__init__()

        # ── ResNet-18 image encoder ───────────────────────────────────────
        resnet = models.resnet18(weights="IMAGENET1K_V1")
        # Freeze all layers
        for p in resnet.parameters():
            p.requires_grad = False
        # Unfreeze layer4 — same strategy that gave 0.7263 on image-only CNN
        for p in resnet.layer4.parameters():
            p.requires_grad = True
        # Remove final FC; after avgpool output is (B, 512, 1, 1)
        self.image_encoder = nn.Sequential(*list(resnet.children())[:-1])

        # ── BioBERT text encoder ──────────────────────────────────────────
        self.text_encoder = AutoModel.from_pretrained(biobert_model_name)
        for p in self.text_encoder.parameters():
            p.requires_grad = False

        # ── Per-stream normalisation (prevents scale mismatch) ────────────
        self.img_norm  = nn.LayerNorm(512)
        self.text_norm = nn.LayerNorm(768)

        # ── Fusion head ───────────────────────────────────────────────────
        self.fusion_head = nn.Sequential(
            nn.Linear(512 + 768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def forward(self, images, input_ids, attn_mask):
        # Image branch
        img_features = self.image_encoder(images)                    # (B, 512, 1, 1)
        img_features = img_features.view(img_features.size(0), -1)   # (B, 512)
        img_features = self.img_norm(img_features)

        # Text branch — mean pooling over non-padding tokens
        bert_out      = self.text_encoder(input_ids=input_ids,
                                          attention_mask=attn_mask)
        hidden        = bert_out.last_hidden_state                    # (B, seq_len, 768)
        mask          = attn_mask.unsqueeze(-1).float()               # (B, seq_len, 1)
        text_features = (hidden * mask).sum(1) / mask.sum(1)         # (B, 768)
        text_features = self.text_norm(text_features)

        # Fusion
        combined = torch.cat([img_features, text_features], dim=1)   # (B, 1280)
        logits   = self.fusion_head(combined)                         # (B, 1)
        return logits


model = ResNetBioBERTFusion().to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Model ready on : {DEVICE}")
print(f"Trainable params: {trainable:,}  /  Total: {total:,}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 144MB/s]


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready on : cuda
Trainable params: 8,724,481  /  Total: 119,817,537


In [8]:
# Cell 8: Debug — verify feature statistics before training
# Both std values should be > 0.05. If either is ~0 there is a preprocessing bug.

model.eval()
sample_batch = next(iter(train_loader))

with torch.no_grad():
    images    = sample_batch["image"].to(DEVICE)
    input_ids = sample_batch["input_ids"].to(DEVICE)
    attn_mask = sample_batch["attn_mask"].to(DEVICE)

    raw_img  = model.image_encoder(images).view(images.size(0), -1)
    bert_out = model.text_encoder(input_ids=input_ids, attention_mask=attn_mask)
    hidden   = bert_out.last_hidden_state
    mask     = attn_mask.unsqueeze(-1).float()
    raw_txt  = (hidden * mask).sum(1) / mask.sum(1)

print(f"img_features  — mean: {raw_img.mean():.4f}  std: {raw_img.std():.4f}")
print(f"text_features — mean: {raw_txt.mean():.4f}  std: {raw_txt.std():.4f}")

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

img_features  — mean: 0.9207  std: 0.9324
text_features — mean: -0.0031  std: 0.3876


In [11]:
# Cell 9: Training setup
EPOCHS    = 30
PATIENCE  = 7
SAVE_PATH = "resnet_biobert_best.pth"

criterion = nn.BCEWithLogitsLoss()

# Separate LR groups: 10x smaller LR for the unfrozen ResNet layer4
optimizer = torch.optim.AdamW([
    {"params": model.fusion_head.parameters(), "lr": 1e-4},
    {"params": model.img_norm.parameters(),    "lr": 1e-4},
    {"params": model.text_norm.parameters(),   "lr": 1e-4},
    {"params": model.image_encoder[-2].parameters(), "lr": 1e-5},  # layer4
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)


def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for batch in tqdm(loader, leave=False):
            images    = batch["image"].to(device)
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attn_mask"].to(device)
            labels    = batch["label"].float().to(device)

            logits = model(images, input_ids, attn_mask).squeeze(1)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds       = (torch.sigmoid(logits) >= 0.5).long()
            correct    += (preds == labels.long()).sum().item()
            total      += labels.size(0)
            total_loss += loss.item() * labels.size(0)

    return total_loss / total, correct / total


print(f"Ready — Epochs: {EPOCHS}  |  Patience: {PATIENCE}  |  Checkpoint: {SAVE_PATH}")

Ready — Epochs: 30  |  Patience: 7  |  Checkpoint: resnet_biobert_best.pth


In [12]:
# Cell 10: Train
best_val_acc   = 0.0
best_weights   = None
patience_count = 0
history        = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion)

    scheduler.step(val_acc)
    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                    "val_loss": val_loss, "val_acc": val_acc})

    improved = val_acc > best_val_acc
    marker   = " checkmark" if improved else ""
    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"Train loss: {train_loss:.4f}  acc: {train_acc:.4f}  "
          f"Val loss: {val_loss:.4f}  acc: {val_acc:.4f}{marker}")

    if improved:
        best_val_acc   = val_acc
        best_weights   = copy.deepcopy(model.state_dict())
        torch.save(best_weights, SAVE_PATH)
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best val acc: {best_val_acc:.4f}")
            break

print(f"\nBest Val Acc: {best_val_acc:.4f}")

Epoch 01/30  Train loss: 0.6699  acc: 0.5933  Val loss: 0.6601  acc: 0.5474 checkmark


Epoch 02/30  Train loss: 0.5963  acc: 0.6880  Val loss: 0.6315  acc: 0.6263 checkmark


Epoch 03/30  Train loss: 0.5288  acc: 0.7587  Val loss: 0.6032  acc: 0.6947 checkmark


Epoch 04/30  Train loss: 0.4641  acc: 0.7987  Val loss: 0.5787  acc: 0.7316 checkmark


Epoch 05/30  Train loss: 0.4173  acc: 0.8200  Val loss: 0.5553  acc: 0.7526 checkmark


...

Epoch 12/30  Train loss: 0.2596  acc: 0.8867  Val loss: 0.5989  acc: 0.7632


Epoch 13/30  Train loss: 0.2774  acc: 0.8813  Val loss: 0.5899  acc: 0.7737


Epoch 14/30  Train loss: 0.2457  acc: 0.9027  Val loss: 0.5945  acc: 0.7737


Epoch 15/30  Train loss: 0.2384  acc: 0.9040  Val loss: 0.6050  acc: 0.7526


Epoch 16/30  Train loss: 0.2386  acc: 0.9000  Val loss: 0.6227  acc: 0.7684
Early stopping at epoch 16. Best val acc: 0.7789

Best Val Acc: 0.7789


In [13]:
# Cell 11: Final evaluation
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
print(f"Loaded best checkpoint: '{SAVE_PATH}'")

model.eval()
all_labels, all_preds, all_probs = [], [], []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating"):
        images    = batch["image"].to(DEVICE)
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attn_mask"].to(DEVICE)
        labels    = batch["label"]

        logits = model(images, input_ids, attn_mask).squeeze(1)
        probs  = torch.sigmoid(logits).cpu()
        preds  = (probs >= 0.5).long()

        all_labels.extend(labels.numpy())
        all_preds.extend(preds.numpy())
        all_probs.extend(probs.numpy())

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, zero_division=0)
auc = roc_auc_score(all_labels, all_probs)

print("\n" + "="*48)
print("  ResNet-18 + BioBERT Fusion")
print("="*48)
print(f"  Accuracy  : {acc:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print(f"  AUC-ROC   : {auc:.4f}")
print("="*48)

Loaded best checkpoint: 'resnet_biobert_best.pth'


Evaluating: 100%|██████████| 12/12 [00:02<00:00,  5.78it/s]


  ResNet-18 + BioBERT Fusion
  Accuracy  : 0.7789
  F1 Score  : 0.7835
  AUC-ROC   : 0.8204


In [15]:
# Cell 12: Comparison table
results = [
    {"Model": "TF-IDF + Logistic Regression",   "Accuracy": 0.6684, "F1": 0.6441, "AUC-ROC": 0.7073},
    {"Model": "TF-IDF + SVM",                    "Accuracy": 0.6895, "F1": 0.6704, "AUC-ROC": 0.7517},
    {"Model": "ResNet-18 (layer4 unfrozen)",      "Accuracy": 0.7263, "F1": 0.7500, "AUC-ROC": 0.7740},
    {"Model": "Fusion: ResNet-18 + CLIP",         "Accuracy": 0.7579, "F1": 0.7723, "AUC-ROC": 0.8194},
    {"Model": "Fusion: ResNet-18 + BioBERT",      "Accuracy": round(acc, 4),
                                                  "F1":       round(f1,  4),
                                                  "AUC-ROC": round(auc, 4)},
]

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

results_df.to_csv("results_v3.csv", index=False)
print("\nSaved to results_v3.csv")

                       Model  Accuracy     F1  AUC-ROC
TF-IDF + Logistic Regression    0.6684 0.6441   0.7073
                TF-IDF + SVM    0.6895 0.6704   0.7517
 ResNet-18 (layer4 unfrozen)    0.7263 0.7500   0.7740
    Fusion: ResNet-18 + CLIP    0.7579 0.7723   0.8194
 Fusion: ResNet-18 + BioBERT    0.7789 0.7835   0.8204

Saved to results_v3.csv


In [16]:
# Cell 13: Save to Google Drive (optional)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import shutil
SAVE_DIR = "/content/drive/MyDrive/Medical_VQA"
os.makedirs(SAVE_DIR, exist_ok=True)

shutil.copy(SAVE_PATH,                os.path.join(SAVE_DIR, SAVE_PATH))
shutil.copy("results_v3.csv", os.path.join(SAVE_DIR, "results_v3.csv"))
print(f"Saved to {SAVE_DIR}")

Mounted at /content/drive
Saved to /content/drive/MyDrive/Medical_VQA
